In [ ]:
import logging
import os
import time
from datetime import datetime, timedelta

import joblib
import numpy as np
import pandas as pd
import requests
import hopsworks

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
)
log = logging.getLogger(__name__)


# CONFIG & FEATURE SCHEMA

LATITUDE = 31.4187
LONGITUDE = 73.0791
CITY_NAME = "Faisalabad"

WEATHER_URL = "https://api.open-meteo.com/v1/forecast"
AQ_URL = "https://air-quality-api.open-meteo.com/v1/air-quality"

BURNING_SEASON_MONTHS = {10, 11}

FINAL_FEATURES = [
    "us_aqi", "us_aqi_lag_1h", "pm25_pm10_ratio",
    "pm2_5", "pm2_5_lag_1h", "pm2_5_roll_mean_6h", "pm2_5_roll_mean_24h",
    "pm2_5_roll_std_24h", "pm10",
    "us_aqi_lag_24h", "us_aqi_lag_48h", "us_aqi_lag_72h",
    "us_aqi_roll_mean_24h", "us_aqi_roll_min_24h", "us_aqi_roll_max_24h", "us_aqi_roll_std_24h",
    "temperature_2m", "relative_humidity_2m", "surface_pressure",
    "wind_speed_10m", "wind_u", "wind_v",
    "is_burning_season",
    "cos_month", "sin_month", "cos_hour", "sin_hour", "is_weekend",
]

MODEL_NAMES = ["aqi_predictor_24h", "aqi_predictor_48h", "aqi_predictor_72h"]



# 1. FETCH RECENT DATA (Past 5 days to compute 72h lags)

def fetch_recent_data(past_days: int = 5) -> pd.DataFrame:
    """Fetches past 5 days of hourly weather + AQ data to satisfy 72h lag requirements."""
    weather_params = {
        "latitude": LATITUDE,
        "longitude": LONGITUDE,
        "past_days": past_days,
        "forecast_days": 1,
        "hourly": [
            "temperature_2m",
            "relative_humidity_2m",
            "surface_pressure",
            "wind_speed_10m",
            "wind_direction_10m",
        ],
        "timezone": "UTC",
    }
    aq_params = {
        "latitude": LATITUDE,
        "longitude": LONGITUDE,
        "past_days": past_days,
        "forecast_days": 1,
        "hourly": ["pm2_5", "pm10", "nitrogen_dioxide", "ozone", "us_aqi"],
        "timezone": "UTC",
    }

    log.info("Fetching past %d days of data for live inference...", past_days)
    res_weather = requests.get(WEATHER_URL, params=weather_params, timeout=30).json()
    res_aq = requests.get(AQ_URL, params=aq_params, timeout=30).json()

    df_weather = pd.DataFrame(res_weather["hourly"])
    df_weather["time"] = pd.to_datetime(df_weather["time"])

    df_aq = pd.DataFrame(res_aq["hourly"])
    df_aq["time"] = pd.to_datetime(df_aq["time"])

    df = pd.merge(df_weather, df_aq, on="time", how="inner")
    df["city"] = CITY_NAME
    return df



# 2. FEATURE ENGINEERING FOR INFERENCE

def engineer_features_for_inference(df: pd.DataFrame) -> pd.DataFrame:
    """Computes spatial, temporal, lag, and rolling features matching training logic."""
    df = df.sort_values("time").reset_index(drop=True)

    # 1. Wind components
    wind_rad = np.radians(df["wind_direction_10m"])
    df["wind_u"] = -df["wind_speed_10m"] * np.sin(wind_rad)
    df["wind_v"] = -df["wind_speed_10m"] * np.cos(wind_rad)

    # 2. Temporal & cyclical features
    df["hour"] = df["time"].dt.hour.astype("int64")
    df["dayofweek"] = df["time"].dt.dayofweek.astype("int64")
    df["month"] = df["time"].dt.month.astype("int64")
    df["is_weekend"] = (df["dayofweek"] >= 5).astype("int64")
    df["is_burning_season"] = df["month"].isin(BURNING_SEASON_MONTHS).astype("int64")

    df["sin_hour"] = np.sin(2 * np.pi * df["hour"] / 24.0)
    df["cos_hour"] = np.cos(2 * np.pi * df["hour"] / 24.0)
    df["sin_month"] = np.sin(2 * np.pi * df["month"] / 12.0)
    df["cos_month"] = np.cos(2 * np.pi * df["month"] / 12.0)

    # 3. Lags
    df["pm2_5_lag_1h"] = df["pm2_5"].shift(1)
    df["us_aqi_lag_1h"] = df["us_aqi"].shift(1)
    df["us_aqi_lag_24h"] = df["us_aqi"].shift(24)
    df["us_aqi_lag_48h"] = df["us_aqi"].shift(48)
    df["us_aqi_lag_72h"] = df["us_aqi"].shift(72)

    # 4. Rolling statistics
    df["pm2_5_roll_mean_6h"] = df["pm2_5"].shift(1).rolling(6).mean()
    df["pm2_5_roll_mean_24h"] = df["pm2_5"].shift(1).rolling(24).mean()
    df["pm2_5_roll_std_24h"] = df["pm2_5"].shift(1).rolling(24).std()

    df["us_aqi_roll_mean_24h"] = df["us_aqi"].shift(1).rolling(24).mean()
    df["us_aqi_roll_std_24h"] = df["us_aqi"].shift(1).rolling(24).std()
    df["us_aqi_roll_min_24h"] = df["us_aqi"].shift(1).rolling(24).min()
    df["us_aqi_roll_max_24h"] = df["us_aqi"].shift(1).rolling(24).max()

    # 5. Composition ratio
    df["pm25_pm10_ratio"] = df["pm2_5"] / (df["pm10"] + 1e-5)

    # Drop early records missing 72h historical context
    df = df.dropna(subset=FINAL_FEATURES).reset_index(drop=True)
    return df

def extract_model_object(loaded_obj, model_name: str):
    """Extracts the estimator containing .predict() if loaded_obj is wrapped in a tuple/list."""
    if hasattr(loaded_obj, "predict"):
        return loaded_obj
    
    if isinstance(loaded_obj, (tuple, list)):
        for item in loaded_obj:
            if hasattr(item, "predict"):
                return item

    raise TypeError(
        f"Loaded artifact for '{model_name}' (type: {type(loaded_obj).__name__}) "
        f"does not contain an object with a .predict() method."
    )


def load_models_from_registry():
    """Fetches the latest version of each model from Hopsworks Model Registry and loads into RAM."""
    log.info("Connecting to Hopsworks Model Registry...")
    project = hopsworks.login()
    mr = project.get_model_registry()

    loaded_models = {}
    for model_name in MODEL_NAMES:
        # Fetch registered versions
        models_list = mr.get_models(name=model_name)
        if not models_list:
            raise ValueError(f"No models registered under name '{model_name}'")

        # Select latest version
        latest_model = max(models_list, key=lambda m: int(m.version))
        version_num = str(latest_model.version)
        log.info("Fetching latest version (%s) of %s...", version_num, model_name)

        # Download directory
        download_path = latest_model.download()

        # Locate model.pkl path dynamically
        model_filepath = None
        for dirpath, _, filenames in os.walk(download_path):
            if "model.pkl" in filenames:
                model_filepath = os.path.join(dirpath, "model.pkl")
                break

        if not model_filepath or not os.path.exists(model_filepath):
            raise FileNotFoundError(
                f"Could not locate 'model.pkl' for model '{model_name}' in '{download_path}'"
            )

        # Load binary artifact
        raw_artifact = joblib.load(model_filepath)
        
        # Unwrap model object if stored as a tuple/list
        loaded_models[model_name] = extract_model_object(raw_artifact, model_name)
        
        log.info(
            "Successfully loaded %s (v%s) into RAM",
            model_name,
            version_num,
        )

    return loaded_models


In [ ]:


# 4. EXECUTE INFERENCE & DISPLAY RESULTS

def main():
    # 1. Ingest recent history & calculate features
    raw_df = fetch_recent_data(past_days=5)
    processed_df = engineer_features_for_inference(raw_df)

    # Extract latest timestamp and corresponding feature vector
    latest_row = processed_df.iloc[-1:]
    current_time = latest_row["time"].values[0]
    current_aqi = latest_row["us_aqi"].values[0]

    X_infer = latest_row[FINAL_FEATURES]

    # 2. Load models from registry into memory
    models = load_models_from_registry()

    # 3. Make predictions across horizons
    pred_24h = float(models["aqi_predictor_24h"].predict(X_infer)[0])
    pred_48h = float(models["aqi_predictor_48h"].predict(X_infer)[0])
    pred_72h = float(models["aqi_predictor_72h"].predict(X_infer)[0])

    # 4. Display formatted forecast
    ts_now = pd.to_datetime(current_time)
    print("\n" + "=" * 55)
    print(f" AIR QUALITY FORECAST FOR {CITY_NAME.upper()}")
    print("=" * 55)
    print(f" Current Time (UTC): {ts_now.strftime('%Y-%m-%d %H:%M:%S')}")
    print(f" Current US AQI:    {current_aqi:.0f}")
    print("-" * 55)
    print(f" +24 Hours ({ts_now + timedelta(hours=24)}): {pred_24h:.1f} AQI")
    print(f" +48 Hours ({ts_now + timedelta(hours=48)}): {pred_48h:.1f} AQI")
    print(f" +72 Hours ({ts_now + timedelta(hours=72)}): {pred_72h:.1f} AQI")
    print("=" * 55 + "\n")


if __name__ == "__main__":
    main()

2026-08-31 23:13:05,013 INFO: Fetching past 5 days of data for live inference...
2026-08-31 23:13:07,879 INFO: Connecting to Hopsworks Model Registry...
2026-08-31 23:13:07,888 INFO: Closing external client and cleaning up certificates.
2026-08-31 23:13:07,893 INFO: Connection closed.
2026-08-31 23:13:07,893 INFO: Initializing external client
2026-08-31 23:13:07,903 INFO: Base URL: https://eu-west.cloud.hopsworks.ai:443
2026-08-31 23:13:13,988 INFO: Python Engine initialized.



Logged in to project, explore it here https://eu-west.cloud.hopsworks.ai:443/p/42119


2026-08-31 23:13:18,957 INFO: Fetching latest version (3) of aqi_predictor_24h...
2026-08-31 23:13:18,967 INFO: Successfully loaded aqi_predictor_24h (v3) into RAM


Using cached model files at 'f:\tmp\hopsworks\models\AQI_Predictor_fsd\aqi_predictor_24h\3\aqi_predictor_24h_3'. Pass local_path or call Model.clear_cache(...) to force a fresh download.


2026-08-31 23:13:20,457 INFO: Fetching latest version (2) of aqi_predictor_48h...
2026-08-31 23:13:20,473 INFO: Successfully loaded aqi_predictor_48h (v2) into RAM


Using cached model files at 'f:\tmp\hopsworks\models\AQI_Predictor_fsd\aqi_predictor_48h\2\aqi_predictor_48h_2'. Pass local_path or call Model.clear_cache(...) to force a fresh download.


2026-08-31 23:13:21,814 INFO: Fetching latest version (2) of aqi_predictor_72h...
2026-08-31 23:13:21,824 INFO: Successfully loaded aqi_predictor_72h (v2) into RAM


Using cached model files at 'f:\tmp\hopsworks\models\AQI_Predictor_fsd\aqi_predictor_72h\2\aqi_predictor_72h_2'. Pass local_path or call Model.clear_cache(...) to force a fresh download.

 AIR QUALITY FORECAST FOR FAISALABAD
 Current Time (UTC): 2026-08-31 23:00:00
 Current US AQI:    151
-------------------------------------------------------
 +24 Hours (2026-09-01 23:00:00): 146.9 AQI
 +48 Hours (2026-09-02 23:00:00): 139.4 AQI
 +72 Hours (2026-09-03 23:00:00): 135.3 AQI

